# 03 · Robustness extras

Lower-priority additions; each can be dropped from the paper if budget is tight.

| Stage | ≈ cost | Answers |
|---|---|---|
| Original-pipeline replication (paper prompt + Mental-Accounting-only KB) | $4 | shows the effect of the original KB bug |
| Stability: 2 extra runs of C3 on the 200 gold reviews | $1 | reproducibility (test-retest κ) |
| Counterfactual focus (behaviour vs situation) | $1.5 | R4.8 + implements Section 3.3.5 |
| External dataset (500 Kindle/Fire/Echo reviews) | $2 | R2.8, R4.9 |

To skip a stage, remove it from `C.PARTS["part3_robustness"]["stages"]` in the next cell before running.

**Safety:** the run stops by itself when `BUDGET_USD` is reached or OpenAI reports no credit. Everything finished is cached; after topping up, re-run the *Run* cell and it continues where it stopped, without paying again for finished reviews.

## Setup

In [ ]:
import os, sys
from pathlib import Path

def find_revision_dir() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, here / "Revision", here.parent / "Revision"]:
        if (cand / "code" / "config.py").exists():
            return cand
    raise FileNotFoundError("Open this notebook from Behavioral_Economics/Revision/notebooks/")

REVISION_DIR = find_revision_dir()
PROJECT_DIR = REVISION_DIR.parent
os.environ["REVISION_DIR"] = str(REVISION_DIR)
os.environ.setdefault("DATASET_DIR", str(PROJECT_DIR / "Dataset"))
os.environ.setdefault("MAX_WORKERS", "8")   # parallel API calls; lower to 2-4 if you hit rate limits
if str(REVISION_DIR / "code") not in sys.path:
    sys.path.insert(0, str(REVISION_DIR / "code"))
print("Revision folder:", REVISION_DIR, "| Python", sys.version.split()[0])

In [ ]:
# Installs any missing packages into THIS kernel (uv, venv or conda). Does nothing if all are present.
import importlib.util, shutil, subprocess
CORE = {"openai": "openai", "pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
        "statsmodels": "statsmodels", "faiss": "faiss-cpu", "pypdf": "pypdf",
        "langchain_text_splitters": "langchain-text-splitters", "openpyxl": "openpyxl",
        "vaderSentiment": "vaderSentiment", "krippendorff": "krippendorff", "lingua": "lingua-language-detector",
        "matplotlib": "matplotlib", "tqdm": "tqdm", "ipywidgets": "ipywidgets"}

def _install(req_file):
    py = sys.executable
    uv = shutil.which("uv") or shutil.which("uv.exe")
    if uv:
        cmd = [uv, "pip", "install", "--python", py, "-r", str(req_file)]
    else:
        if importlib.util.find_spec("pip") is None:
            subprocess.run([py, "-m", "ensurepip", "--upgrade"], check=True)
        cmd = [py, "-m", "pip", "install", "-r", str(req_file)]
    print("Running:", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout or "")[-1500:], (r.stderr or "")[-1500:]); r.check_returncode()

missing = [pkg for mod, pkg in CORE.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing); _install(REVISION_DIR / "requirements.txt"); importlib.invalidate_caches()
print("Missing packages:", [p for m, p in CORE.items() if importlib.util.find_spec(m) is None] or "none")

In [ ]:
import importlib, json
import pandas as pd
import config as C                     # also loads ../.env
import pipeline as PL
C.ensure_dirs()
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)
MOCK = os.environ.get("MOCK_LLM") == "1"   # code testing only - never for results
print("OpenAI key in use :", C.key_fingerprint(), "| loaded from:", C.ENV_LOADED or "environment (no .env found)")
print("If this is not the key you expect: fix Behavioral_Economics/.env, then Restart the kernel and re-run.")

## Cost check (free)

In [ ]:
rem = PL.remaining_estimate(["part3_robustness"])
display(rem[["stage", "namespace", "calls_cached", "calls_left", "pct_done", "usd_spent_so_far", "usd_left_indicative"]])
LEFT = rem["usd_left_indicative"].sum()
print(f"Indicative cost to finish this part: ${LEFT:.2f}  (gpt-4o $2.50/$10, gpt-4o-mini $0.15/$0.60 per 1M tokens)")

In [ ]:
# Hard cap on NEW spend in this run. Set it to what you can afford right now;
# if it is lower than the cost above, the part simply stops early and resumes next time.
BUDGET_USD = 10

In [ ]:
# Optional connectivity check (free request)
if not MOCK:
    from openai import OpenAI
    try:
        ids = [m.id for m in OpenAI().models.list().data]
        print("OpenAI reachable | gpt-4o:", C.MAIN_MODEL in ids, "| gpt-4o-mini:", C.DRAFTER_MODEL in ids)
    except Exception as e:
        print("OpenAI NOT reachable:", repr(e)[:300])

In [ ]:
# Optional: choose which robustness stages to run (default: all)
C.PARTS["part3_robustness"]["stages"] = ["kb", "original", "stability", "cf", "external"]
print(C.PARTS["part3_robustness"]["stages"])

## Run (paid)

In [ ]:
runner = PL.Runner(mock=MOCK)
result = runner.run_part("part3_robustness", BUDGET_USD)
result

## Results so far (free: re-computed from whatever has been run)

In [ ]:
import analysis, figures
importlib.reload(analysis); importlib.reload(figures)
analysis.run(); figures.run()
print((C.RESULTS_DIR / "06_analysis" / "STATUS.md").read_text(encoding="utf-8"))

In [ ]:
A = C.RESULTS_DIR / "06_analysis"
def show(f, **kw):
    p = A / f
    if p.exists():
        print("\n###", f); display(pd.read_csv(p, **kw))

show("C_test_retest_stability.csv")
show("E_cf_focus_tests.csv")
show("G_generalisability.csv")
show("A_prevalence_by_system.csv")

In [ ]:
# What has been paid for so far, and what each part still costs (reads the cache only - free)
display(PL.remaining_summary())